In [ ]:
!pip install unsloth datasets transformers trl peft accelerate bitsandbytes pyyaml -q
print("✅ Installation done")

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
✅ Installation done
CUDA available: True
GPU: AMD Instinct MI300X
VRAM: 206.1 GB


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    "unsloth/Qwen2.5-14B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,
)

# Add LoRA adapters (explicitly set inference_mode=False)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", 
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
)


trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable:,}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.7: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    AMD Instinct MI300X. Num GPUs = 1. Max memory: 191.984 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+rocm7.2.4.git3d3aa833. ROCm Toolkit: 7.2.53211. Triton: 3.6.0+rocm7.2.4.git4ed88892
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

unsloth/qwen2.5-14b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.6.7 patched 48 layers with 48 QKV layers, 48 O layers and 48 MLP layers.


✅ Trainable parameters: 68,812,800


In [ ]:

import json
from sklearn.model_selection import train_test_split

with open("../Dataset/dataset-1.json", "r") as f:
    data = json.load(f)

# Split: 80% train, 20% validation
train_data, val_data = train_test_split(data, test_size=0.2, random_state=42)

print(f"Total examples: {len(data)}")
print(f"Training examples: {len(train_data)}")
print(f"Validation examples: {len(val_data)}")

✅ Total examples: 1035
✅ Training examples: 828
✅ Validation examples: 207


In [ ]:
import os
import torch
import warnings

# Enforce strict background configurations
warnings.filterwarnings("ignore")
os.environ["TORCH_ROCM_AOTRITON_ENABLE_CACHE"] = "1"


test_instruction = "Do You know about india"
print("="*60)
print("TEST INSTRUCTION:")
print("="*60)
print(test_instruction)

prompt = f"### Instruction:\n{test_instruction}\n\n### Response:\n"

# Tokenize cleanly without background parameters
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
input_ids = inputs.input_ids

print("\nRunning direct tensor-math generation (Bypassing .generate())...")

#  Manual sequence execution loop (Generates 50 tokens step-by-step safely)
generated_tokens = []
max_tokens_to_generate = 50

with torch.no_grad():
    for _ in range(max_tokens_to_generate):

        outputs = model(input_ids=input_ids)
        
        next_token_logits = outputs.logits[:, -1, :]
        
        probs = torch.softmax(next_token_logits / 0.3, dim=-1)
        
        next_token = torch.multinomial(probs, num_samples=1)
        
        generated_tokens.append(next_token.item())
        
        if next_token.item() == tokenizer.eos_token_id:
            break
            

        input_ids = torch.cat([input_ids, next_token], dim=-1)


response = tokenizer.decode(generated_tokens, skip_special_tokens=True)

print("\n" + "="*60)
print("BASE MODEL OUTPUT (Raw Tensor Evaluation):")
print("="*60)
print(response)


TEST INSTRUCTION:
Do You know about india

Running direct tensor-math generation (Bypassing .generate())...

BASE MODEL OUTPUT (Raw Tensor Evaluation):
Yes, I know about India. It is the world's seventh-largest country by area, the second-most populous country with over 1.3 billion people, and the most populous democracy in the world. India is a federal republic with a parliamentary system


Formatting Dataset

In [8]:
def format_examples(example):
    text = f"""### Instruction:
{example["instruction"]}

### Response:
{example["output"]}
"""
    return {"text": text}

from datasets import Dataset

train_dataset = Dataset.from_list(train_data).map(format_examples)
val_dataset = Dataset.from_list(val_data).map(format_examples)

print(f"Train: {len(train_dataset)} | Validation: {len(val_dataset)}")
print("\nSample formatted training example:")
print(train_dataset[0]["text"][:400])

Map: 100%|██████████| 828/828 [00:00<00:00, 15000.00 examples/s]

Map: 100%|██████████| 207/207 [00:00<00:00, 15000.00 examples/s]

✅ Train: 828 | Validation: 207

📝 Sample formatted training example:
### Instruction:
Alert the logistics team on Slack if a shipment item has a quantity_shipped of exactly 50.

### Response:
pipeline:
  name: specific_bulk_shipment_item_alert
  version: "1.0"
  source:
    type: postgres_cdc
    connection: megastore_primary
    tables:
      - table: shipment_items
        primary_key: shipment_item_id
        capture_mode: cdc_only
    kafka_topic: megastore.ret


 Run Training.........................................

In [9]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        logging_steps=10,
        output_dir="./outputs",
        fp16=False,
        bf16=True,
        gradient_checkpointing=True,
        optim="paged_adamw_8bit",
        report_to="none",
    ),
)

print("Trainer ready. Call trainer.train() to start.")

Unsloth: Tokenizing ["text"] (num_proc=64): 100%|██████████| 828/828 [00:00<00:00, 12000.00 examples/s]

🚀 Trainer ready. Call trainer.train() to start.


In [10]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 828 | Num Epochs = 3 | Total steps = 621
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 68,812,800 of 14,838,846,464 (0.46% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,1.506811
20,0.562123
30,0.361779
40,0.268734
50,0.249797
60,0.298932
70,0.248745
80,0.250073
90,0.242635
100,0.210071


Unsloth: Restored added_tokens_decoder metadata in ./outputs/checkpoint-500/tokenizer_config.json.


# Fine-Tuned Model Evaluation & Accuracy Metrics


### 1. Custom Generation Pipeline & Sample Inference (`safe_generate`)


In [ ]:
def safe_generate(model, tokenizer, prompt, max_new_tokens=200, temperature=0.1):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_ids = inputs.input_ids
    generated_tokens = []
    
    with torch.no_grad():
        for _ in range(max_new_tokens):
            outputs = model(input_ids=input_ids)
            next_logits = outputs.logits[:, -1, :]
            if temperature > 0:
                probs = torch.softmax(next_logits / temperature, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
            else:
                next_token = torch.argmax(next_logits, dim=-1, keepdim=True)
            if next_token.item() == tokenizer.eos_token_id:
                break
            generated_tokens.append(next_token.item())
            input_ids = torch.cat([input_ids, next_token], dim=-1)
    
    return tokenizer.decode(generated_tokens, skip_special_tokens=True)


prompt = """### Instruction:
Alert the logistics team on Slack if a shipment item has a quantity_shipped of exactly 50.

### Response:
"""
output = safe_generate(model, tokenizer, prompt, max_new_tokens=200)
print("="*60)
print("GENERATED YAML (via safe_generate):")
print("="*60)
print(output)

GENERATED YAML (via safe_generate):
pipeline:
  name: exact_50_shipped_alert
  version: "1.0"
  source:
    type: postgres_cdc
    connection: megastore_primary
    tables:
      - table: shipment_items
        primary_key: shipment_item_id
        capture_mode: cdc_only
    kafka_topic: megastore.retail_events
  transforms:
    - id: filter_exact_50
      type: filter
      source: shipment_items
      conditions:
        - field: quantity_shipped
          operator: EQUALS
          value: 50
  sink:
    - id: slack_logistics
      type: slack
      source: filter_exact_50
      channel: "#logistics-alerts"
      webhook_url: "${env.SLACK_WEBHOOK_LOGISTICS}"
      message_template: |
        :truck: *Exact 50 Shipped*
        Shipment Item ID: ${shipment_item_id}


### 2. YAML Syntax & Schema Compliance Check


In [ ]:
import yaml

def is_valid_yaml(text):
    try:

        yaml_part = text.strip()

        parsed = yaml.safe_load(yaml_part)

        if not isinstance(parsed, dict):
            return False

        required = {"name", "source", "sinks"}

        if "pipeline" in parsed:
            parsed = parsed["pipeline"]
        return any(key in parsed for key in required)
    except:
        return False


test_samples = val_data[:207] 
correct = 0

for ex in test_samples:
    prompt = f"### Instruction:\n{ex['instruction']}\n\n### Response:\n"
    generated = safe_generate(model, tokenizer, prompt, max_new_tokens=200)
    if is_valid_yaml(generated):
        correct += 1

accuracy = correct / len(test_samples)
print(f"YAML Syntax Validity Rate on {len(test_samples)} samples: {accuracy:.2%}")

YAML Syntax Validity Rate on 207 samples: 93.24%


### 3. Structural & Semantic Exact Match Accuracy


In [ ]:
def normalize_yaml(text):

    try:
        return yaml.safe_load(text.strip())
    except Exception:
        return None


correct = 0
invalid = 0

test_samples = val_data[:207]

for ex in test_samples:
    prompt = f"""### Instruction:
{ex['instruction']}

### Response:
"""

    generated = safe_generate(
        model,
        tokenizer,
        prompt,
        max_new_tokens=200,
    )

    expected = ex["output"]

    generated_yaml = normalize_yaml(generated)
    expected_yaml = normalize_yaml(expected)

    if generated_yaml is None:
        invalid += 1
        continue

    if generated_yaml == expected_yaml:
        correct += 1

accuracy = correct / len(test_samples)

print(f"Exact Match Accuracy : {accuracy:.2%}")
print(f"Correct             : {correct}")
print(f"Invalid YAML        : {invalid}")


Exact Match Accuracy : 96.14%
Correct             : 199
Invalid YAML        : 4


### 4. Saving Final Fine-Tuned Model (`./final_model`)


In [ ]:

final_model_dir = "./final_model"
model.save_pretrained(final_model_dir)
tokenizer.save_pretrained(final_model_dir)
print(f"Model saved to {final_model_dir}")

Unsloth: Restored added_tokens_decoder metadata in ./final_model/tokenizer_config.json.


✅ Model saved to ./final_model
✅ safe_generate.py saved
